# Tinker SFT Notebook for the Local Bluesky Dataset

This notebook discovers the dataset already present in this folder, converts it into Tinker-friendly chat examples, and runs a LoRA supervised fine-tuning loop plus held-out evaluation.

Defaults are intentionally conservative:
- `SMOKE_TEST = True` keeps the run short and cheap.
- `REQUESTED_MODELS` starts with `gpt-oss-20b`.
- The notebook resolves those names against the live Tinker server capabilities and skips anything unsupported.

Before running:
1. `pip install -r requirements.txt`
2. Copy `.env.example` to `.env`
3. Set `TINKER_API_KEY` in `.env` or your shell


In [ ]:
from __future__ import annotations

import time
import traceback
import os
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from IPython.display import clear_output, display

import tinker
from tinker import AdamParams
from tinker_cookbook.renderers import TrainOnWhat

from tinker_experiment_manager import (
    build_dataset_variant_summary_df,
    build_experiment_dataset_variants,
    build_experiment_plan_df,
    resolve_selected_experiment_specs,
)
from tinker_notebook_diagnostics import (
    extract_notebook_artifacts,
    list_recent_training_runs_df,
    recovered_tracking_state,
    session_id_from_run_id,
    unique_preserving_order,
)
from tinker_notebook_env import (
    describe_tinker_api_key,
    ensure_tinker_api_key,
)
from tinker_notebook_resume import (
    build_resume_selection,
    display_resume_selector,
    list_checkpoints_df,
    list_resumable_runs_df,
)
from tinker_notebook_workflows import (
    format_seconds,
    run_training_loop as helper_run_training_loop,
    save_named_sampler_checkpoint,
    save_named_state_checkpoint,
)
from tinker_stop_control import (
    clear_stop_request,
    default_stop_signal_path,
    format_stop_request,
    read_stop_request,
    request_stop,
)
from tinker_training_utils import (
    build_batches,
    build_datums,
    build_eval_prompts,
    build_post_examples,
    compute_batch_loss,
    dataset_summary,
    evaluate_cross_entropy,
    find_dataset_root,
    load_dataset_bundle,
    maybe_take_examples,
    resolve_model_names,
    sample_generations,
    select_renderer_name,
    slugify_name,
)

_ = load_dotenv()
TINKER_API_KEY_INFO = ensure_tinker_api_key(required=True)
print(describe_tinker_api_key(TINKER_API_KEY_INFO))


In [ ]:
#put your data in the workspace root and etc here
WORKSPACE_ROOT = Path.cwd().resolve()
DATASET_ROOT = find_dataset_root(WORKSPACE_ROOT)
STOP_SIGNAL_PATH = default_stop_signal_path(WORKSPACE_ROOT)
CLEAR_STOP_SIGNAL_BEFORE_RUN = True

# Gemma is intentionally disabled here because your current Tinker server does not expose a matching model.
REQUESTED_MODELS = ["gpt-oss-20b"]
SELECTED_MODEL_ALIAS = "gpt-oss-20b"
RUN_ONE_MODEL_ONLY = True

# Override keys can be either the requested alias or the resolved provider/model name.
MODEL_OVERRIDES = {
    "gpt-oss-20b": {"learning_rate": 1e-4, "renderer_name": None},
    "openai/gpt-oss-20b": {"learning_rate": 1e-4, "renderer_name": None},
}

SMOKE_TEST = False
SHUFFLE_SEED = 7
RUN_BASELINE_EVAL = False
RUN_POST_TRAIN_EVAL = True
RUN_POST_TRAIN_SAMPLING = True
KEEP_SYSTEM_AWAKE_DURING_TRAINING = True
KEEP_MONITOR_FOCUSED_ON_CURRENT_RUN = True
LIVE_MONITOR_REFRESH_SECONDS = 15
LORA_RANK = 16
BATCH_SIZE = 8
MAX_LENGTH = 512
NUM_EPOCHS = 1 if SMOKE_TEST else 3
TRAIN_EXAMPLE_LIMIT = 96 if SMOKE_TEST else None
MAX_STEPS_PER_MODEL = 12 if SMOKE_TEST else None
EVAL_EXAMPLE_LIMIT = 5
SAMPLE_TEMPERATURE = 0.71
SAMPLE_MAX_TOKENS = 96
PRINT_EVERY_STEPS = 1 if SMOKE_TEST else 5
SAVE_STATE_EVERY_STEPS = 4 if SMOKE_TEST else 25
RESUME_ADDITIONAL_STEPS = 8 if SMOKE_TEST else 40
STATE_CHECKPOINT_PREFIX = "tinker-studio-state"
SAMPLER_CHECKPOINT_PREFIX = "tinker-studio-sampler"

DEFAULT_TRAINING_CONFIG = {
    "model_alias": SELECTED_MODEL_ALIAS,
    "lora_rank": LORA_RANK,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "num_epochs": NUM_EPOCHS,
    "train_example_limit": TRAIN_EXAMPLE_LIMIT,
    "max_steps_per_model": MAX_STEPS_PER_MODEL,
    "eval_example_limit": EVAL_EXAMPLE_LIMIT,
    "sample_temperature": SAMPLE_TEMPERATURE,
    "sample_max_tokens": SAMPLE_MAX_TOKENS,
    "print_every_steps": PRINT_EVERY_STEPS,
    "save_state_every_steps": SAVE_STATE_EVERY_STEPS,
    "run_post_train_eval": RUN_POST_TRAIN_EVAL,
    "run_post_train_sampling": RUN_POST_TRAIN_SAMPLING,
    "keep_system_awake_during_training": KEEP_SYSTEM_AWAKE_DURING_TRAINING,
    "learning_rate": float(MODEL_OVERRIDES.get(SELECTED_MODEL_ALIAS, {}).get("learning_rate", 1e-4)),
}

EXPERIMENT_SPECS = [
    {
        "run_name": "essay_recent_r16",
        "model_alias": SELECTED_MODEL_ALIAS,
        "dataset_variant": "recent_posts_plus_essays",
        "lora_rank": LORA_RANK,
        "learning_rate": 1e-4,
        "batch_size": BATCH_SIZE,
        "max_length": MAX_LENGTH,
        "num_epochs": NUM_EPOCHS,
        "notes": "Original training posts plus later posts and chunked essays. Eval is disabled by default for this variant.",
    },
    {
        "run_name": "initial_r32_lr7e5_b6",
        "model_alias": SELECTED_MODEL_ALIAS,
        "dataset_variant": "initial_posts",
        "lora_rank": 32,
        "learning_rate": 7e-5,
        "batch_size": 6,
        "max_length": MAX_LENGTH,
        "num_epochs": NUM_EPOCHS,
        "notes": "Initial post-only split with higher LoRA rank, lower learning rate, and smaller batch size.",
    },
]
RUN_EXPERIMENT_NAMES = [spec["run_name"] for spec in EXPERIMENT_SPECS]

print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Dataset root:   {DATASET_ROOT}")
print(f"Stop signal:   {STOP_SIGNAL_PATH}")
print(f"Smoke test:     {SMOKE_TEST}")
print(f"Selected model: {SELECTED_MODEL_ALIAS}")
print(f"Epochs:         {NUM_EPOCHS}")
max_steps_label = MAX_STEPS_PER_MODEL if MAX_STEPS_PER_MODEL is not None else "all epoch steps"
print(f"Max steps:      {max_steps_label}")
print(f"Print every:    {PRINT_EVERY_STEPS} step(s)")
print(f"Save state every: {SAVE_STATE_EVERY_STEPS} step(s)")
print(f"Resume addl steps: {RESUME_ADDITIONAL_STEPS}")
print(f"Keep awake:     {KEEP_SYSTEM_AWAKE_DURING_TRAINING}")
print(f"Clear stop before run: {CLEAR_STOP_SIGNAL_BEFORE_RUN}")
print(f"Experiment runs: {', '.join(RUN_EXPERIMENT_NAMES)}")


In [ ]:
#dataloader
bundle = load_dataset_bundle(DATASET_ROOT)
dataset_variants = build_experiment_dataset_variants(bundle)
dataset_variant_summary_df = build_dataset_variant_summary_df(dataset_variants)
display(dataset_variant_summary_df)

selected_experiment_specs = resolve_selected_experiment_specs(
    EXPERIMENT_SPECS,
    RUN_EXPERIMENT_NAMES,
)
experiment_plan_df = build_experiment_plan_df(
    selected_experiment_specs,
    dataset_variants,
    default_config=DEFAULT_TRAINING_CONFIG,
)
display(experiment_plan_df)

split_summary = pd.DataFrame(
    [
        {"split": "train", **dataset_summary(bundle.train_rows)},
        {"split": "validation", **dataset_summary(bundle.validation_rows)},
        {"split": "test", **dataset_summary(bundle.test_rows)},
    ]
)
display(split_summary)

initial_variant = dataset_variants["initial_posts"]
train_examples = initial_variant.train_examples
validation_examples = initial_variant.validation_examples
test_examples = initial_variant.test_examples

preview_df = pd.DataFrame(
    [
        {
            "example_id": example.example_id,
            "opening_text": example.opening_text,
            "target_text": example.target_text,
        }
        for example in train_examples[:5]
    ]
)
display(preview_df)


In [ ]:
#check for float  target models
service_client = tinker.ServiceClient()
rest_client = service_client.create_rest_client()
capabilities = service_client.get_server_capabilities()
supported_models = [model.model_name for model in capabilities.supported_models]

resolutions = resolve_model_names(REQUESTED_MODELS, supported_models)
resolution_df = pd.DataFrame([vars(item) for item in resolutions])
display(resolution_df)

selected_resolution = None
if SELECTED_MODEL_ALIAS:
    selected_resolution = next(
        (
            item
            for item in resolutions
            if item.requested_name == SELECTED_MODEL_ALIAS or item.resolved_name == SELECTED_MODEL_ALIAS
        ),
        None,
    )
if selected_resolution is None:
    selected_resolution = next((item for item in resolutions if item.resolved_name), None)

if selected_resolution and selected_resolution.resolved_name:
    print(
        "Selected run target:",
        f"{selected_resolution.requested_name} -> {selected_resolution.resolved_name}",
    )
else:
    print("No supported model resolved yet. Fix the model mapping before starting a run.")

relevant_supported_models = [
    model_name
    for model_name in supported_models
    if any(token in model_name.lower() for token in ("gpt", "qwen", "llama"))
]
relevant_supported_models[:25]


In [ ]:
def get_model_override(requested_name: str, resolved_name: str) -> dict:
    merged = {}
    merged.update(MODEL_OVERRIDES.get(requested_name, {}))
    merged.update(MODEL_OVERRIDES.get(resolved_name, {}))
    return merged


def get_resolution_for_alias(model_alias: str | None):
    if not model_alias:
        return selected_resolution
    resolution = next(
        (
            item
            for item in resolutions
            if item.requested_name == model_alias or item.resolved_name == model_alias
        ),
        None,
    )
    if resolution is None:
        raise ValueError(f"Could not resolve model alias: {model_alias}")
    return resolution


def get_experiment_config(experiment_spec: dict, requested_name: str, resolved_name: str) -> dict:
    variant_name = str(experiment_spec.get("dataset_variant") or "initial_posts")
    if variant_name not in dataset_variants:
        raise KeyError(f"Unknown dataset_variant: {variant_name}")

    variant = dataset_variants[variant_name]
    model_override = get_model_override(requested_name, resolved_name)
    renderer_name = select_renderer_name(
        resolved_name,
        override=experiment_spec.get("renderer_name", model_override.get("renderer_name")),
    )
    run_post_train_eval = bool(
        experiment_spec.get(
            "run_post_train_eval",
            RUN_POST_TRAIN_EVAL and variant.allow_post_train_eval,
        )
    )
    return {
        "run_name": str(experiment_spec["run_name"]),
        "dataset_variant_name": variant_name,
        "dataset_variant": variant,
        "notes": str(experiment_spec.get("notes") or variant.notes),
        "renderer_name": renderer_name,
        "lora_rank": int(experiment_spec.get("lora_rank", LORA_RANK)),
        "learning_rate": float(experiment_spec.get("learning_rate", model_override.get("learning_rate", 1e-4))),
        "batch_size": int(experiment_spec.get("batch_size", BATCH_SIZE)),
        "max_length": int(experiment_spec.get("max_length", MAX_LENGTH)),
        "num_epochs": int(experiment_spec.get("num_epochs", NUM_EPOCHS)),
        "train_example_limit": experiment_spec.get("train_example_limit", TRAIN_EXAMPLE_LIMIT),
        "max_steps_per_model": experiment_spec.get("max_steps_per_model", MAX_STEPS_PER_MODEL),
        "eval_example_limit": int(experiment_spec.get("eval_example_limit", EVAL_EXAMPLE_LIMIT)),
        "sample_temperature": float(experiment_spec.get("sample_temperature", SAMPLE_TEMPERATURE)),
        "sample_max_tokens": int(experiment_spec.get("sample_max_tokens", SAMPLE_MAX_TOKENS)),
        "print_every_steps": int(experiment_spec.get("print_every_steps", PRINT_EVERY_STEPS)),
        "save_state_every_steps": int(experiment_spec.get("save_state_every_steps", SAVE_STATE_EVERY_STEPS)),
        "run_post_train_eval": run_post_train_eval,
        "run_post_train_sampling": bool(experiment_spec.get("run_post_train_sampling", RUN_POST_TRAIN_SAMPLING)),
    }


def get_current_session_id(client):
    holder = getattr(client, "holder", None)
    return holder.get_session_id() if holder else None


def get_sampling_session_id(sampling_client):
    return getattr(sampling_client, "_sampling_session_id", None)


def get_session_snapshot(session_id):
    if not session_id:
        return {"session_id": None, "training_run_ids": [], "sampler_ids": []}
    session_info = rest_client.get_session(session_id).result()
    return {
        "session_id": session_id,
        "training_run_ids": list(session_info.training_run_ids),
        "sampler_ids": list(session_info.sampler_ids),
    }


def get_run_monitor_df(training_run_ids):
    rows = []
    now_utc = pd.Timestamp.now(tz="UTC")
    for training_run_id in unique_preserving_order(list(training_run_ids)):
        if not training_run_id:
            continue
        run = rest_client.get_training_run(training_run_id).result()
        checkpoints = rest_client.list_checkpoints(training_run_id).result()
        last_request_time = getattr(run, "last_request_time", None)
        seconds_since_last_request = None
        if last_request_time is not None:
            seconds_since_last_request = round(
                (now_utc - pd.Timestamp(last_request_time)).total_seconds(),
                1,
            )
        rows.append(
            {
                "training_run_id": run.training_run_id,
                "session_id": session_id_from_run_id(run.training_run_id),
                "base_model": run.base_model,
                "lora_rank": run.lora_rank,
                "corrupted": run.corrupted,
                "last_request_time_utc": str(last_request_time),
                "seconds_since_last_request": seconds_since_last_request,
                "num_checkpoints": len(checkpoints.checkpoints),
                "last_checkpoint": getattr(run.last_checkpoint, "checkpoint_id", None),
                "last_sampler_checkpoint": getattr(run.last_sampler_checkpoint, "checkpoint_id", None),
            }
        )
    return pd.DataFrame(rows)


def get_session_monitor_df(session_ids):
    rows = []
    for session_id in unique_preserving_order(list(session_ids)):
        if not session_id:
            continue
        session_info = rest_client.get_session(session_id).result()
        rows.append(
            {
                "session_id": session_id,
                "training_run_ids": list(session_info.training_run_ids),
                "sampler_ids": list(session_info.sampler_ids),
            }
        )
    return pd.DataFrame(rows)


def get_sampler_monitor_df(sampler_ids):
    rows = []
    for sampler_id in unique_preserving_order(list(sampler_ids)):
        if not sampler_id:
            continue
        sampler_info = rest_client.get_sampler(sampler_id).result()
        rows.append(
            {
                "sampler_id": sampler_info.sampler_id,
                "base_model": sampler_info.base_model,
                "model_path": sampler_info.model_path,
            }
        )
    return pd.DataFrame(rows)


def run_model_experiment(resolution, experiment_spec: dict):
    requested_name = resolution.requested_name
    resolved_name = resolution.resolved_name
    if not resolved_name:
        raise ValueError(f"No resolved model for {requested_name}: {resolution.note}")

    experiment_config = get_experiment_config(experiment_spec, requested_name, resolved_name)
    dataset_variant = experiment_config["dataset_variant"]
    training_run_label = slugify_name(experiment_config["run_name"]) or "fresh"

    print()
    print(
        f"[START] run_name={experiment_config['run_name']} dataset_variant={experiment_config['dataset_variant_name']} "
        f"requested={requested_name} resolved={resolved_name}"
    )
    print(
        f"[START] rank={experiment_config['lora_rank']} lr={experiment_config['learning_rate']:.2e} "
        f"batch_size={experiment_config['batch_size']} epochs={experiment_config['num_epochs']} "
        f"max_length={experiment_config['max_length']}"
    )
    print(f"[START] notes={experiment_config['notes']}")
    if not experiment_config["run_post_train_eval"]:
        print("[START] post-train eval is disabled for this run.")
    print("[START] creating LoRA training client")

    session_id = get_current_session_id(service_client)
    training_client = service_client.create_lora_training_client(
        base_model=resolved_name,
        rank=experiment_config["lora_rank"],
    )
    session_id = get_current_session_id(service_client) or session_id
    training_run_id = training_client.model_id
    print(f"[READY] session_id={session_id}")
    print(f"[READY] training_run_id={training_run_id}")
    print(f"[READY] renderer={experiment_config['renderer_name']}")
    print("[PLAN] Leave this notebook kernel running while training. Use the monitor cell or monitor_tinker_runs.py from a second process if you want live polling.")

    result = helper_run_training_loop(
        training_client=training_client,
        requested_name=requested_name,
        resolved_name=resolved_name,
        renderer_name=experiment_config["renderer_name"],
        learning_rate=experiment_config["learning_rate"],
        training_run_label=training_run_label,
        additional_steps=experiment_config["max_steps_per_model"],
        train_examples=dataset_variant.train_examples,
        validation_examples=dataset_variant.validation_examples,
        test_examples=dataset_variant.test_examples,
        train_example_limit=experiment_config["train_example_limit"],
        shuffle_seed=SHUFFLE_SEED,
        batch_size=experiment_config["batch_size"],
        max_length=experiment_config["max_length"],
        num_epochs=experiment_config["num_epochs"],
        print_every_steps=experiment_config["print_every_steps"],
        save_state_every_steps=experiment_config["save_state_every_steps"],
        eval_example_limit=experiment_config["eval_example_limit"],
        sample_temperature=experiment_config["sample_temperature"],
        sample_max_tokens=experiment_config["sample_max_tokens"],
        run_post_train_eval=experiment_config["run_post_train_eval"],
        run_post_train_sampling=experiment_config["run_post_train_sampling"],
        keep_system_awake=KEEP_SYSTEM_AWAKE_DURING_TRAINING,
        stop_signal_path=STOP_SIGNAL_PATH,
        state_checkpoint_prefix=STATE_CHECKPOINT_PREFIX,
        sampler_checkpoint_prefix=SAMPLER_CHECKPOINT_PREFIX,
        slugify_name=slugify_name,
        get_sampling_session_id=get_sampling_session_id,
    )

    result["summary"].update(
        {
            "run_name": experiment_config["run_name"],
            "dataset_variant": experiment_config["dataset_variant_name"],
            "dataset_notes": dataset_variant.notes,
            "source_counts": dataset_variant.source_counts,
            "session_id": session_id,
            "training_run_id": training_run_id,
            "baseline_training_run_id": None,
            "pre_validation_nll": float("nan"),
            "pre_test_nll": float("nan"),
            "lora_rank": experiment_config["lora_rank"],
            "learning_rate": experiment_config["learning_rate"],
            "batch_size": experiment_config["batch_size"],
            "max_length": experiment_config["max_length"],
            "num_epochs": experiment_config["num_epochs"],
            "max_steps_per_model": experiment_config["max_steps_per_model"],
            "run_post_train_eval": experiment_config["run_post_train_eval"],
            "run_post_train_sampling": experiment_config["run_post_train_sampling"],
        }
    )
    result["run_monitor"] = get_run_monitor_df([training_run_id])
    result["session_monitor"] = get_session_monitor_df([session_id]) if session_id else pd.DataFrame()
    if result["summary"].get("sampler_id"):
        result["sampler_monitor"] = get_sampler_monitor_df([result["summary"]["sampler_id"]])
    print(f"[DONE] run_name={experiment_config['run_name']} training_run_id={training_run_id}")
    print(f"[DONE] final_state_path={result['summary'].get('final_state_path')}")
    if result["summary"].get("sampler_model_path"):
        print(f"[DONE] sampler_model_path={result['summary']['sampler_model_path']}")
    return result


def resume_model_experiment(resume_config, requested_name=None, resolved_name=None):
    if not resume_config or not resume_config.get("tinker_path"):
        raise ValueError("resume_config with a valid tinker_path is required")

    resume_tinker_path = resume_config["tinker_path"]
    resolved_name = resolved_name or resume_config.get("base_model")
    requested_name = requested_name or resume_config.get("requested_name") or resolved_name
    if not resolved_name:
        raise ValueError("Could not determine resolved_name for resumed run")

    override = get_model_override(requested_name, resolved_name)
    renderer_name = select_renderer_name(
        resolved_name,
        override=override.get("renderer_name"),
    )
    learning_rate = float(override.get("learning_rate", 1e-4))

    print()
    print(f"[RESUME] source_run_id={resume_config.get('training_run_id')}")
    print(f"[RESUME] source_checkpoint={resume_config.get('checkpoint_id')}")
    print(f"[RESUME] loading optimizer state from {resume_tinker_path}")
    training_client = service_client.create_training_client_from_state_with_optimizer(resume_tinker_path)
    session_id = get_current_session_id(service_client)
    resumed_training_run_id = training_client.model_id
    print(f"[RESUME] new_session_id={session_id}")
    print(f"[RESUME] new_training_run_id={resumed_training_run_id}")
    print(f"[RESUME] renderer={renderer_name} learning_rate={learning_rate:.2e}")

    result = helper_run_training_loop(
        training_client=training_client,
        requested_name=requested_name,
        resolved_name=resolved_name,
        renderer_name=renderer_name,
        learning_rate=learning_rate,
        training_run_label="resume",
        additional_steps=RESUME_ADDITIONAL_STEPS,
        train_examples=train_examples,
        validation_examples=validation_examples,
        test_examples=test_examples,
        train_example_limit=TRAIN_EXAMPLE_LIMIT,
        shuffle_seed=SHUFFLE_SEED,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        num_epochs=NUM_EPOCHS,
        print_every_steps=PRINT_EVERY_STEPS,
        save_state_every_steps=SAVE_STATE_EVERY_STEPS,
        eval_example_limit=EVAL_EXAMPLE_LIMIT,
        sample_temperature=SAMPLE_TEMPERATURE,
        sample_max_tokens=SAMPLE_MAX_TOKENS,
        run_post_train_eval=RUN_POST_TRAIN_EVAL,
        run_post_train_sampling=RUN_POST_TRAIN_SAMPLING,
        keep_system_awake=KEEP_SYSTEM_AWAKE_DURING_TRAINING,
        stop_signal_path=STOP_SIGNAL_PATH,
        state_checkpoint_prefix=STATE_CHECKPOINT_PREFIX,
        sampler_checkpoint_prefix=SAMPLER_CHECKPOINT_PREFIX,
        slugify_name=slugify_name,
        get_sampling_session_id=get_sampling_session_id,
    )

    result["summary"].update(
        {
            "session_id": session_id,
            "training_run_id": resumed_training_run_id,
            "resumed_from_training_run_id": resume_config.get("training_run_id"),
            "resumed_from_checkpoint_id": resume_config.get("checkpoint_id"),
            "resumed_from_tinker_path": resume_tinker_path,
        }
    )
    result["run_monitor"] = get_run_monitor_df([resumed_training_run_id])
    result["session_monitor"] = get_session_monitor_df([session_id]) if session_id else pd.DataFrame()
    if result["summary"].get("sampler_id"):
        result["sampler_monitor"] = get_sampler_monitor_df([result["summary"]["sampler_id"]])
    return result


In [ ]:
#### initialize tracking
runs = globals().get("runs", {})
run_summaries = globals().get("run_summaries", [])
NOTEBOOK_PATH = WORKSPACE_ROOT / "tinker_train_and_eval.ipynb"
MANUAL_TRACKED_RUN_IDS = globals().get("MANUAL_TRACKED_RUN_IDS", [])
MANUAL_TRACKED_SESSION_IDS = globals().get("MANUAL_TRACKED_SESSION_IDS", [])
MANUAL_TRACKED_SAMPLER_IDS = globals().get("MANUAL_TRACKED_SAMPLER_IDS", [])


In [ ]:
#stop controls
print(f"Reliable stop while training is running: double-click {WORKSPACE_ROOT / 'stop_tinker_training.bat'}")
print(f"Clear a stale stop request: {WORKSPACE_ROOT / 'clear_tinker_stop_request.bat'}")
print("Notebook buttons below only work when the kernel is idle.")

stop_status = widgets.HTML()
stop_output = widgets.Output()


def refresh_stop_controls():
    stop_status.value = f"<pre>{format_stop_request(STOP_SIGNAL_PATH)}</pre>"


def on_request_stop(_):
    payload = request_stop(STOP_SIGNAL_PATH, reason="notebook_widget")
    with stop_output:
        clear_output(wait=True)
        print(f"Stop requested: {payload['path']}")
    refresh_stop_controls()


def on_clear_stop(_):
    removed = clear_stop_request(STOP_SIGNAL_PATH)
    with stop_output:
        clear_output(wait=True)
        if removed:
            print(f"Cleared stop request: {STOP_SIGNAL_PATH}")
        else:
            print(f"No stop request existed: {STOP_SIGNAL_PATH}")
    refresh_stop_controls()


stop_button = widgets.Button(description="Request Stop", button_style="danger", icon="stop")
clear_button = widgets.Button(description="Clear Stop", icon="check")
refresh_button = widgets.Button(description="Refresh", icon="refresh")

stop_button.on_click(on_request_stop)
clear_button.on_click(on_clear_stop)
refresh_button.on_click(lambda _: refresh_stop_controls())
refresh_stop_controls()

display(widgets.VBox([
    widgets.HBox([stop_button, clear_button, refresh_button]),
    stop_status,
    stop_output,
]))


In [ ]:
#diagnostics and recovery
notebook_artifacts_df, notebook_errors_df = extract_notebook_artifacts(NOTEBOOK_PATH)
recovered_state = recovered_tracking_state(NOTEBOOK_PATH)
recovered_training_run_ids = recovered_state["training_run_ids"]
recovered_sampler_ids = recovered_state["sampler_ids"]
recovered_session_ids = recovered_state["session_ids"]

api_recent_runs_df = list_recent_training_runs_df(rest_client, limit=20)
api_recent_training_run_ids = (
    api_recent_runs_df["training_run_id"].dropna().tolist() if not api_recent_runs_df.empty else []
)
api_recent_session_ids = unique_preserving_order(
    api_recent_runs_df["session_id"].dropna().tolist() if not api_recent_runs_df.empty else []
)

display(
    notebook_artifacts_df
    if not notebook_artifacts_df.empty
    else pd.DataFrame([{"note": "No saved run IDs found in the notebook JSON yet."}])
)
display(
    notebook_errors_df
    if not notebook_errors_df.empty
    else pd.DataFrame([{"note": "No saved notebook errors found in the notebook JSON."}])
)
display(
    api_recent_runs_df
    if not api_recent_runs_df.empty
    else pd.DataFrame([{"note": "No training runs were returned by the API."}])
)

print("Recovered training run IDs from notebook JSON:", recovered_training_run_ids)
print("Recovered sampler IDs from notebook JSON:", recovered_sampler_ids)
print("Recovered session IDs from notebook JSON:", recovered_session_ids)
print("Recent training run IDs from API:", api_recent_training_run_ids)
print("Recent session IDs from API:", api_recent_session_ids)
print("If seconds_since_last_request keeps increasing across refreshes, the run is idle.")


In [ ]:
#resume selector
resume_ui = display_resume_selector(rest_client, limit=20)
selected_resume_config = None
if resume_ui.get("selection") is not None:
    selected_resume_config = {
        "training_run_id": resume_ui["selection"].training_run_id,
        "checkpoint_id": resume_ui["selection"].checkpoint_id,
        "tinker_path": resume_ui["selection"].tinker_path,
        "base_model": resume_ui["selection"].base_model,
        "lora_rank": resume_ui["selection"].lora_rank,
        "status": resume_ui["selection"].status,
    }

selected_resume_config


In [ ]:
#resume run
if not selected_resume_config:
    raise RuntimeError("No selected_resume_config found. Run the resume selector cell first.")

if CLEAR_STOP_SIGNAL_BEFORE_RUN:
    if clear_stop_request(STOP_SIGNAL_PATH):
        print(f"Cleared stale stop request: {STOP_SIGNAL_PATH}")
else:
    pending_stop_request = read_stop_request(STOP_SIGNAL_PATH)
    if pending_stop_request:
        raise RuntimeError(f"Clear the pending stop request first: {format_stop_request(STOP_SIGNAL_PATH)}")

resolved_name_for_resume = selected_resume_config.get("base_model")
requested_name_for_resume = selected_resolution.requested_name if selected_resolution else resolved_name_for_resume

print(
    f"Resuming from {selected_resume_config['training_run_id']} @ {selected_resume_config['checkpoint_id']}"
)

try:
    resumed_run = resume_model_experiment(
        selected_resume_config,
        requested_name=requested_name_for_resume,
        resolved_name=resolved_name_for_resume,
    )
    runs = {f"resumed::{resolved_name_for_resume}": resumed_run}
    run_summaries = [resumed_run["summary"]]
    MANUAL_TRACKED_RUN_IDS = unique_preserving_order(
        list(MANUAL_TRACKED_RUN_IDS) + [resumed_run["summary"].get("training_run_id")]
    )
    MANUAL_TRACKED_SESSION_IDS = unique_preserving_order(
        list(MANUAL_TRACKED_SESSION_IDS) + [resumed_run["summary"].get("session_id")]
    )
    MANUAL_TRACKED_SAMPLER_IDS = unique_preserving_order(
        list(MANUAL_TRACKED_SAMPLER_IDS) + [resumed_run["summary"].get("sampler_id")]
    )
except Exception as exc:
    session_id = get_current_session_id(service_client)
    error_summary = {
        "requested_name": requested_name_for_resume,
        "resolved_name": resolved_name_for_resume,
        "status": "resume_error",
        "error_type": type(exc).__name__,
        "error": repr(exc),
        "session_id": session_id,
        "resumed_from_training_run_id": selected_resume_config.get("training_run_id"),
        "resumed_from_checkpoint_id": selected_resume_config.get("checkpoint_id"),
    }
    if session_id:
        try:
            session_snapshot = get_session_snapshot(session_id)
            MANUAL_TRACKED_RUN_IDS = unique_preserving_order(
                list(MANUAL_TRACKED_RUN_IDS) + list(session_snapshot.get("training_run_ids") or [])
            )
            MANUAL_TRACKED_SESSION_IDS = unique_preserving_order(
                list(MANUAL_TRACKED_SESSION_IDS) + [session_id]
            )
            MANUAL_TRACKED_SAMPLER_IDS = unique_preserving_order(
                list(MANUAL_TRACKED_SAMPLER_IDS) + list(session_snapshot.get("sampler_ids") or [])
            )
        except Exception as session_exc:
            error_summary["session_lookup_error"] = repr(session_exc)
    run_summaries = [error_summary]
    print(traceback.format_exc())

results_df = pd.DataFrame(run_summaries)
display(results_df)


In [ ]:
#begin run
if not selected_experiment_specs:
    raise RuntimeError("No experiment specs selected. Check RUN_EXPERIMENT_NAMES in the config cell.")

if CLEAR_STOP_SIGNAL_BEFORE_RUN:
    if clear_stop_request(STOP_SIGNAL_PATH):
        print(f"Cleared stale stop request: {STOP_SIGNAL_PATH}")
else:
    pending_stop_request = read_stop_request(STOP_SIGNAL_PATH)
    if pending_stop_request:
        raise RuntimeError(f"Clear the pending stop request first: {format_stop_request(STOP_SIGNAL_PATH)}")

runs = {}
run_summaries = []

for experiment_spec in selected_experiment_specs:
    pending_stop_request = read_stop_request(STOP_SIGNAL_PATH)
    if pending_stop_request:
        print(f"[SCHEDULE] stop already requested; not starting {experiment_spec['run_name']}")
        print(format_stop_request(STOP_SIGNAL_PATH))
        break

    experiment_resolution = get_resolution_for_alias(experiment_spec.get("model_alias"))
    if experiment_resolution is None or not experiment_resolution.resolved_name:
        raise RuntimeError(
            f"No resolved model is selected for {experiment_spec['run_name']}. Re-run the capabilities cell and fix the model selection first."
        )

    print()
    print("=" * 88)
    print(
        f"Attempting run: {experiment_spec['run_name']} -> "
        f"{experiment_resolution.requested_name} / {experiment_resolution.resolved_name}"
    )
    print("=" * 88)

    try:
        run = run_model_experiment(experiment_resolution, experiment_spec)
        runs[experiment_spec["run_name"]] = run
        run_summaries.append(run["summary"])
        MANUAL_TRACKED_RUN_IDS = unique_preserving_order(
            list(MANUAL_TRACKED_RUN_IDS)
            + [run["summary"].get("training_run_id"), run["summary"].get("baseline_training_run_id")]
        )
        MANUAL_TRACKED_SESSION_IDS = unique_preserving_order(
            list(MANUAL_TRACKED_SESSION_IDS) + [run["summary"].get("session_id")]
        )
        MANUAL_TRACKED_SAMPLER_IDS = unique_preserving_order(
            list(MANUAL_TRACKED_SAMPLER_IDS) + [run["summary"].get("sampler_id")]
        )
        if run["summary"].get("stop_requested"):
            print(f"[SCHEDULE] stop requested during {experiment_spec['run_name']}; skipping remaining experiments.")
            break
    except Exception as exc:
        session_id = get_current_session_id(service_client)
        error_summary = {
            "run_name": experiment_spec["run_name"],
            "dataset_variant": experiment_spec.get("dataset_variant"),
            "requested_name": experiment_resolution.requested_name if experiment_resolution else None,
            "resolved_name": experiment_resolution.resolved_name if experiment_resolution else None,
            "status": "error",
            "error_type": type(exc).__name__,
            "error": repr(exc),
            "session_id": session_id,
            "training_run_ids": None,
            "sampler_ids": None,
        }
        if session_id:
            try:
                session_snapshot = get_session_snapshot(session_id)
                error_summary["training_run_ids"] = session_snapshot.get("training_run_ids")
                error_summary["sampler_ids"] = session_snapshot.get("sampler_ids")
                MANUAL_TRACKED_RUN_IDS = unique_preserving_order(
                    list(MANUAL_TRACKED_RUN_IDS) + list(session_snapshot.get("training_run_ids") or [])
                )
                MANUAL_TRACKED_SESSION_IDS = unique_preserving_order(
                    list(MANUAL_TRACKED_SESSION_IDS) + [session_id]
                )
                MANUAL_TRACKED_SAMPLER_IDS = unique_preserving_order(
                    list(MANUAL_TRACKED_SAMPLER_IDS) + list(session_snapshot.get("sampler_ids") or [])
                )
            except Exception as session_exc:
                error_summary["session_lookup_error"] = repr(session_exc)
        run_summaries.append(error_summary)
        print(traceback.format_exc())

results_df = pd.DataFrame(run_summaries)
display(results_df)


In [ ]:
#monitor
runs = globals().get("runs", {})
run_summaries = globals().get("run_summaries", [])
MANUAL_TRACKED_RUN_IDS = globals().get("MANUAL_TRACKED_RUN_IDS", [])
MANUAL_TRACKED_SESSION_IDS = globals().get("MANUAL_TRACKED_SESSION_IDS", [])
MANUAL_TRACKED_SAMPLER_IDS = globals().get("MANUAL_TRACKED_SAMPLER_IDS", [])


def tracked_run_ids_from_runs(runs_dict):
    tracked = []
    for run in runs_dict.values():
        summary = run["summary"]
        for key in ("training_run_id", "baseline_training_run_id"):
            value = summary.get(key)
            if value:
                tracked.append(value)
    return tracked


def tracked_session_ids_from_runs(runs_dict):
    tracked = []
    for run in runs_dict.values():
        session_id = run["summary"].get("session_id")
        if session_id:
            tracked.append(session_id)
    return tracked


def tracked_sampler_ids_from_runs(runs_dict):
    tracked = []
    for run in runs_dict.values():
        sampler_id = run["summary"].get("sampler_id")
        if sampler_id:
            tracked.append(sampler_id)
    return tracked


def current_tracking_state(api_limit=10):
    recovered_state_local = recovered_tracking_state(NOTEBOOK_PATH)
    api_recent_runs_df = list_recent_training_runs_df(rest_client, limit=api_limit)
    focused_run_ids = unique_preserving_order(
        tracked_run_ids_from_runs(runs) + list(MANUAL_TRACKED_RUN_IDS)
    )

    if KEEP_MONITOR_FOCUSED_ON_CURRENT_RUN and focused_run_ids:
        api_recent_runs_df = api_recent_runs_df[
            api_recent_runs_df["training_run_id"].isin(focused_run_ids)
        ].reset_index(drop=True)

    api_recent_training_run_ids = (
        api_recent_runs_df["training_run_id"].dropna().tolist() if not api_recent_runs_df.empty else []
    )
    api_recent_session_ids = (
        api_recent_runs_df["session_id"].dropna().tolist() if not api_recent_runs_df.empty else []
    )

    if focused_run_ids:
        tracked_run_ids = focused_run_ids
    elif recovered_state_local["training_run_ids"]:
        tracked_run_ids = recovered_state_local["training_run_ids"]
    else:
        tracked_run_ids = api_recent_training_run_ids

    tracked_session_ids = unique_preserving_order(
        tracked_session_ids_from_runs(runs)
        + list(MANUAL_TRACKED_SESSION_IDS)
        + [session_id_from_run_id(run_id) for run_id in tracked_run_ids]
    )
    if not tracked_session_ids:
        tracked_session_ids = unique_preserving_order(
            recovered_state_local["session_ids"] + api_recent_session_ids
        )

    tracked_sampler_ids = unique_preserving_order(
        tracked_sampler_ids_from_runs(runs)
        + list(MANUAL_TRACKED_SAMPLER_IDS)
    )
    if not tracked_sampler_ids:
        tracked_sampler_ids = unique_preserving_order(recovered_state_local["sampler_ids"])

    return {
        "training_run_ids": tracked_run_ids,
        "session_ids": tracked_session_ids,
        "sampler_ids": tracked_sampler_ids,
        "api_recent_runs_df": api_recent_runs_df,
    }


def display_tracking_state(tracking_state):
    print("Focused training run IDs:", tracking_state["training_run_ids"])
    print("Focused session IDs:", tracking_state["session_ids"])
    if tracking_state["sampler_ids"]:
        print("Focused sampler IDs:", tracking_state["sampler_ids"])
    if tracking_state["training_run_ids"]:
        display(get_run_monitor_df(tracking_state["training_run_ids"]))
    if tracking_state["session_ids"]:
        display(get_session_monitor_df(tracking_state["session_ids"]))
    if tracking_state["sampler_ids"]:
        display(get_sampler_monitor_df(tracking_state["sampler_ids"]))
    if not tracking_state["api_recent_runs_df"].empty:
        display(tracking_state["api_recent_runs_df"])
    print("If seconds_since_last_request keeps increasing across refreshes, the run is idle.")


def live_monitor_runs(refresh_seconds=LIVE_MONITOR_REFRESH_SECONDS, iterations=12):
    for refresh_index in range(iterations):
        clear_output(wait=True)
        tracking_state = current_tracking_state()
        display_tracking_state(tracking_state)
        print(f"Refresh {refresh_index + 1}/{iterations} at {pd.Timestamp.now(tz='UTC').isoformat()}")
        if refresh_index + 1 < iterations:
            time.sleep(refresh_seconds)


tracking_state = current_tracking_state()
has_tracking = bool(
    tracking_state["training_run_ids"]
    or tracking_state["session_ids"]
    or tracking_state["sampler_ids"]
    or not tracking_state["api_recent_runs_df"].empty
)
if has_tracking:
    display_tracking_state(tracking_state)

    if run_summaries:
        display(pd.DataFrame(run_summaries))

    if runs:
        fig, ax = plt.subplots(figsize=(10, 5))
        for model_name, run in runs.items():
            history_df = run["history"]
            if history_df.empty:
                continue
            ax.plot(
                history_df["step"],
                history_df["train_loss"],
                marker="o",
                linewidth=2,
                label=model_name,
            )
        ax.set_title("Training Loss by Step")
        ax.set_xlabel("Step")
        ax.set_ylabel("Cross-entropy loss")
        ax.legend()
        ax.grid(alpha=0.25)
        plt.show()
    else:
        print("No in-memory loss history exists in this kernel yet.")
else:
    print("No tracked runs yet. Run the diagnostics cell above or start a training run.")


In [ ]:
runs = globals().get("runs", {})

if not runs:
    print("No in-memory sample outputs exist in this kernel yet. Start a training run to populate this view.")
else:
    for model_name, run in runs.items():
        print(f"\n=== {model_name} ===")
        display(run["samples"][["opening_text", "target_text", "generated_text"]])
